<a href="https://colab.research.google.com/github/Nathanamarante/ResistoMap/blob/main/ResistoMap.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ResistoMap

---

Objetivo: Mapeamento de genes de resistência a antibióticos presentes na Salmonella enterica

# Checklist

Etapas para elaboração:

3. Selecionar e obter os dados

CARD (card.mcmaster.ca): banco de referência com sequências e ontologia dos genes de resistência.
NCBI Pathogen Detection / NCBI Datasets: genomas e metadados (espécie, país, ano, fonte do isolado), com genes AMR já anotados em muitos casos.

ResFinder / AMRFinderPlus: bases e ferramentas alternativas para comparação.

Documentem os critérios de seleção (espécie, qualidade do assembly, período).

### *CHECKLIST: X (Baixado CARD (Referência) e NCBI)

4. Preparar o ambiente
Python com Biopython, pandas, numpy, matplotlib/seaborn, scipy e, opcionalmente, networkx e scikit-learn. Usem Jupyter ou Colab e um repositório no GitHub.

### *CHECKLIST: X

5. Pré-processamento e controle de qualidade

Filtrar genomas por completude, número de contigs e metadados faltantes.
Padronizar nomes de espécies, países e datas.
Remover duplicatas.

### *CHECKLIST: A FAZER

6. Identificar os genes de resistência
Há duas abordagens, e pode-se combinar ambas:

Usar ferramentas prontas (RGI do CARD ou AMRFinderPlus) e processar a saída em Python.
Implementar uma busca própria: alinhar ou procurar as sequências do CARD nos genomas (por exemplo, com BLAST local e filtros de identidade e cobertura). Isso reforça o lado algorítmico da disciplina.

### *CHECKLIST: A FAZER

7. Construir a matriz presença/ausência
Montem uma tabela genomas × genes (0/1), a base de todas as análises seguintes.

### *CHECKLIST: A FAZER

8. Análises exploratórias e estatísticas

Prevalência de cada gene e de cada classe de antibiótico.
Número médio de genes de resistência por genoma.
Comparação entre grupos (países, anos, fontes: hospitalar, ambiental, animal), com testes estatísticos como qui-quadrado.
Coocorrência de genes (correlação ou índice de Jaccard) e rede de coocorrência.
Agrupamento hierárquico de genomas por perfil de resistência.

### *CHECKLIST: A FAZER

9. Análise avançada (opcional)

PCA ou UMAP sobre a matriz de perfis.
Modelo de machine learning para prever o fenótipo (resistente/sensível) a partir dos genes, com validação cruzada.
Identificação de genomas multirresistentes (MDR).

### *CHECKLIST: A FAZER

10. Visualização dos resultados
Barras de prevalência, heatmap de presença/ausência, mapa ou série temporal, rede de coocorrência e dendrograma.

### *CHECKLIST: A FAZER

11. Interpretação biológica e limitações
Relacionem os achados com a literatura e com a saúde pública. Discutam vieses: genomas de hospitais estão super-representados, e a presença do gene não garante expressão ou resistência fenotípica.

### *CHECKLIST: A FAZER

12. Documentação e reprodutibilidade
README, notebook organizado, lista de dependências e instruções para reproduzir.

### *CHECKLIST: A FAZER

13. Preparar o seminário
Roteiro sugerido: problema → dados → método → resultados → conclusões e limitações. Reservem tempo para ensaiar e dividam as falas entre os integrantes.

### *CHECKLIST: A FAZER

Critérios de seleção dos dados:
Salmonella enterica

Localização: Brazil

Qualidade do Assembly: Não filtrado (Revisar durante o desenvolvimento)

Período: sem restrição de data

_______________________________________________________________________________
Arquivos do CARD Dataset (Referência) são genomas (FASTA). As variantes no NCBI estão em FNA)

_______________________________________________________________________________

Decisão sobre cada arquivo utilizado:
Espécie diferente de Salmonella enterica =	Excluir

Arquivo vazio ou corrompido =	Excluir

Tamanho muito abaixo de 3,5 Mb =	Excluir ou revisar

País faltando, mas análise será por país =	Excluir ou colocar em grupo “desconhecido”

Ano faltando, mas análise será temporal =	Excluir ou colocar em grupo “desconhecido”

Duplicata exata =	Manter apenas uma

Duplicata quase idêntica =	Escolher a de melhor qualidade

______________________________________________________

# Rodar todas as bibliotecas primeiro para não dar erro no código

# Adicionando bibiotecas e Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
%cd /content/drive/MyDrive/Colab Notebooks/ResistoMap


/content/drive/MyDrive/Colab Notebooks/ResistoMap


In [ ]:
!pip install biopython pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 35.9 MB/s  0:00:00


In [ ]:
# Módulo OS
import os

#Módulo Sys
import sys

# Módulo CSV
import csv

# Módulo RE
import re

# Argparse
import argparse

# Hashlib
import hashlib

# Json
import json

# Datetime
from datetime import datetime


# Biblioteca para bioinformática (manipulação de arquivos FASTA, GenBank, etc.)
import Bio
from Bio import SeqIO, Align

# Bibliotecas para manipulação e análise de dados estruturados
import pandas as pd
import numpy as np

# Bibliotecas para análise estatística e matemática
import scipy
from scipy import stats

# Bibliotecas para visualização de dados
import matplotlib.pyplot as plt
import seaborn as sns

# Bibliotecas opcionais para grafos/redes e Machine Learning
import networkx as nx
import sklearn

# Confirmando as versões instaladas
print("Biopython versão:", Bio.__version__)
print("Pandas versão:", pd.__version__)

Biopython versão: 1.88
Pandas versão: 2.2.3


______________________________________________________

# Leitura Metadados/CSV CARD (Referência)
- Vamos ter que fazer leitura dos arquivos FASTA e desenvolver o arquivo com metadados CSV

In [ ]:
%cd /content/drive/MyDrive/Colab Notebooks/ResistoMap/CARD Dataset (Referência)

/content/drive/MyDrive/Colab Notebooks/ResistoMap/CARD Dataset (Referência)


In [ ]:
'''
Código abaixo foi desenvolvido para ler os arquivos de resistência a antimicrobianos
disponíveis em formato FASTA, baixados do card.mcmaster.ca utilizado como referência
E gerar um arquivo chamado ids_card_extraidos.csv para buscar os metadados no site do CARD

'''
# ============================================================================
''' #REMOVA PARA VOLTAR O CÓDIGO ABAIXO
# --- CONFIGURAÇÃO ---
# Coloque o caminho da pasta onde estão os arquivos FASTA a serem trabalhados
caminho_da_pasta = "/content/drive/MyDrive/Colab Notebooks/ResistoMap/CARD Dataset (Referência)"
nome_do_csv = "ids_card_extraidos.csv" #Nome do arquivo csv gerado ao final

def extrair_ids_card(pasta_origem, arquivo_saida):
    """
    Lê todos os arquivos FASTA em uma pasta e extrai os IDs únicos.

    Args:
        pasta_origem (str): Caminho da pasta onde estão os arquivos .fasta
        arquivo_saida (str): Nome do arquivo CSV de saída
    """
    ids_encontrados = set()  # Usamos um set para garantir que os IDs sejam únicos

    # Listar arquivos na pasta
    arquivos = [f for f in os.listdir(pasta_origem) if f.endswith(('.fasta', '.fa', '.fna'))]

    if not arquivos:
        print(f"Nenhum arquivo .fasta encontrado na pasta: {pasta_origem}")
        return

    print(f"Iniciando leitura de {len(arquivos)} arquivos...")

    for nome_arquivo in arquivos:
        caminho_completo = os.path.join(pasta_origem, nome_arquivo)
        try:
            # SeqIO.parse lê o arquivo e retorna um iterador de registros
            for registro in SeqIO.parse(caminho_completo, "fasta"):
                # O .id geralmente contém o ID do CARD (ex: CARD_000001)
                # Se o ID estiver no cabeçalho mas não no .id, usamos registro.description
                id_sequencia = registro.id
                ids_encontrados.add(id_sequencia)
        except Exception as e:
            print(f"Erro ao ler o arquivo {nome_arquivo}: {e}")

    # Converter o set para uma lista ordenada para salvar no CSV
    lista_ids = sorted(list(ids_encontrados))

    # Salvar em CSV
    try:
        with open(arquivo_saida, mode='w', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow(['CARD_ID']) # Cabeçalho
            for id_item in lista_ids:
                writer.writerow([id_item])

        print(f"Sucesso! {len(lista_ids)} IDs únicos foram extraídos.")
        print(f"Os dados foram salvos em: {arquivo_saida}")
    except Exception as e:
        print(f"Erro ao salvar o arquivo CSV: {e}")

# Executar a função
extrair_ids_card(caminho_da_pasta, nome_do_csv)


Iniciando leitura de 13 arquivos...
Sucesso! 11 IDs únicos foram extraídos.
Os dados foram salvos em: ids_card_extraidos.csv


In [ ]:
%cd /content/drive/MyDrive/Colab Notebooks/ResistoMap/CARD Dataset (Referência)/Cruzamento de IDs extraídos

"""
Cruzamento de IDs extraídos (ids_card_extraidos.csv) com os metadados do CARD
(aro_index.tsv + nucleotide_fasta_protein_homolog_model.fasta).

Se os arquivos ainda não estiverem no Colab, vai aparecer o botão de upload.
Saída: metadados_card_final.csv (baixado automaticamente no Colab).
"""
''' #REMOVA PARA VOLTAR O CÓDIGO ABAIXO
# =============================== CONFIGURAÇÃO ===============================
ARQ_IDS = "ids_card_extraidos.csv"
ARQ_INDEX = "aro_index.tsv"
ARQ_FASTA = "nucleotide_fasta_protein_homolog_model.fasta"
ARQ_SAIDA = "metadados_card_final.csv"

SEP_SAIDA = ","            # use ";" se for abrir direto no Excel em português
TOLERANCIA_COORD = 1       # CARD usa início 0-based; seus IDs podem usar 1-based
INCLUIR_COLUNAS_AUDITORIA = False   # True = acrescenta tipo/confiança/observação

# Colunas de metadados que vão para o CSV final (nomes do aro_index.tsv)
COLUNAS_METADADOS = [
    "ARO Accession", "ARO Name", "CARD Short Name",
    "Protein Accession", "DNA Accession",
    "AMR Gene Family", "Drug Class", "Resistance Mechanism",
]

# Desempate manual para IDs de nucleotídeo cujo accession tem vários modelos no
# índice e cuja coordenada não consta no FASTA (modelos de variante/mutação).
# Chave = ID original; valor = trecho do "ARO Name" que identifica o gene.
# Atribuições inferidas pelo tamanho do intervalo; confirme no NCBI.
RESOLUCAO_MANUAL = {
    "gb|AE006468.2|2373710-2376346|Salmonella": "gyrA",
    "gb|AE006468.2|3336954-3339212|Salmonella": "parC",
}
# ============================================================================


def garantir_arquivos():
    """No Colab, abre o upload se algum arquivo estiver faltando."""
    faltando = [a for a in (ARQ_IDS, ARQ_INDEX, ARQ_FASTA) if not os.path.exists(a)]
    if not faltando:
        return
    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError(f"Arquivos não encontrados na pasta atual: {faltando}")
    print("Envie os arquivos:", faltando)
    files.upload()  # grava na pasta atual do Colab
    faltando = [a for a in faltando if not os.path.exists(a)]
    if faltando:
        raise FileNotFoundError(
            f"Ainda faltam: {faltando}. Se o Colab renomeou (ex.: 'arquivo (1).tsv'), "
            "ajuste os nomes na seção CONFIGURAÇÃO."
        )


def sem_versao(acc):
    return acc.split(".")[0]


def interpretar_id(cid):
    """Extrai accession, intervalo (ou '') e ARO (ou '') de um ID do tipo
    gb|ACESSO|[+|-|]inicio-fim|[ARO:xxxx|]Nome."""
    partes = [p.strip() for p in cid.split("|")]
    acc = partes[1] if len(partes) > 1 else partes[0]
    intervalo = next((p for p in partes if re.fullmatch(r"\d+-\d+", p)), "")
    aro = next((p for p in partes if p.startswith("ARO:")), "")
    return acc, intervalo, aro


def ler_fasta_headers(caminho):
    """Retorna {accession: [(inicio, fim, fita, ARO), ...]} a partir dos cabeçalhos."""
    mapa = {}
    with open(caminho, encoding="utf-8", errors="replace") as fh:
        for linha in fh:
            if not linha.startswith(">"):
                continue
            p = linha[1:].strip().split("|")
            if len(p) < 5 or not re.fullmatch(r"\d+-\d+", p[3]):
                continue
            ini, fim = map(int, p[3].split("-"))
            mapa.setdefault(p[1], []).append((ini, fim, p[2], p[4]))
    return mapa


# --------------------------------- carga ------------------------------------
garantir_arquivos()

ids = pd.read_csv(ARQ_IDS, dtype=str)
col_id = "CARD_ID" if "CARD_ID" in ids.columns else ids.columns[0]
ids = ids[[col_id]].rename(columns={col_id: "CARD_ID"})
ids["CARD_ID"] = ids["CARD_ID"].str.strip()
ids = ids.dropna().drop_duplicates().reset_index(drop=True)

idx = pd.read_csv(ARQ_INDEX, sep="\t", dtype=str).fillna("")
fasta = ler_fasta_headers(ARQ_FASTA)

# Proteínas citadas na própria lista (ajudam a desempatar accessions de DNA)
prot_na_lista = {
    interpretar_id(c)[0] for c in ids["CARD_ID"] if not interpretar_id(c)[1]
}

print(f"IDs de entrada: {len(ids)} | modelos no aro_index: {len(idx)} | "
      f"accessions com coordenada no FASTA: {len(fasta)}\n")


# ------------------------------- cruzamento ---------------------------------
def buscar(cid):
    """Retorna (DataFrame de modelos, tipo, confiança, observação)."""
    acc, intervalo, aro = interpretar_id(cid)
    base = sem_versao(acc)

    # 1) o próprio ID já traz o ARO
    if aro:
        m = idx[idx["ARO Accession"] == aro]
        if len(m):
            return m, "ARO no ID", "Alta", ""

    # 2) ID de proteína (sem intervalo)
    if not intervalo:
        m = idx[idx["Protein Accession"] == acc]
        if len(m):
            obs = f"Ligado a {len(m)} modelos ARO." if len(m) > 1 else ""
            return m, "Protein Accession", "Alta", obs
        m = idx[idx["Protein Accession"].map(sem_versao) == base]
        if len(m):
            return m, "Protein Accession (sem versão)", "Média", "Versão do accession difere."
        m = idx[idx["DNA Accession"].map(sem_versao) == base]
        if len(m):
            return m, "DNA Accession", "Média", "Sem intervalo; todos os modelos do accession."
        return idx.iloc[0:0], "SEM CORRESPONDÊNCIA", "", "Accession não consta no aro_index."

    # 3) ID de nucleotídeo com intervalo: tenta casar a coordenada no FASTA
    ini, fim = map(int, intervalo.split("-"))
    for f_ini, f_fim, fita, f_aro in fasta.get(acc, []):
        if abs(f_ini - ini) <= TOLERANCIA_COORD and f_fim == fim:
            m = idx[idx["ARO Accession"] == f_aro]
            if len(m):
                return m, "Coordenada no FASTA", "Alta", f"FASTA: {f_ini}-{f_fim} (fita {fita})."

    # 4) DNA Accession no índice
    m = idx[idx["DNA Accession"] == acc]
    if m.empty:
        m = idx[idx["DNA Accession"].map(sem_versao) == base]
    if m.empty:
        return idx.iloc[0:0], "SEM CORRESPONDÊNCIA", "", "Accession não consta no aro_index."
    if len(m) == 1:
        return m, "DNA Accession (único)", "Alta", f"Único modelo para {acc}."

    # 4b) vários modelos: desempate manual
    gene = RESOLUCAO_MANUAL.get(cid)
    if gene:
        mm = m[m["ARO Name"].str.contains(gene, case=False, regex=False)]
        if len(mm):
            return mm, "DNA Accession + manual", "Média", \
                f"{len(m)} modelos em {acc}; atribuído a '{gene}' manualmente. Confirmar."

    # 4c) vários modelos: usa proteínas presentes na lista
    mm = m[m["Protein Accession"].isin(prot_na_lista)]
    if 0 < len(mm) < len(m):
        return mm, "DNA Accession + proteínas da lista", "Baixa", \
            f"{len(m)} modelos em {acc}; restrito às proteínas da sua lista. AMBÍGUO."
    return m, "DNA Accession (ambíguo)", "Baixa", f"{len(m)} modelos em {acc}. AMBÍGUO."


resultados = []
for cid in ids["CARD_ID"]:
    m, tipo, conf, obs = buscar(cid)
    if m.empty:
        linha = {"CARD_ID": cid, **{c: "" for c in COLUNAS_METADADOS}}
        linha.update(Tipo_correspondencia=tipo, Confianca=conf, Observacao=obs)
        resultados.append(linha)
        continue
    for _, r in m.iterrows():
        linha = {"CARD_ID": cid, **{c: r[c] for c in COLUNAS_METADADOS}}
        linha.update(Tipo_correspondencia=tipo, Confianca=conf, Observacao=obs)
        resultados.append(linha)

final = pd.DataFrame(resultados).drop_duplicates().reset_index(drop=True)

# ------------------------------- relatório ----------------------------------
print("Resumo por tipo de correspondência:")
print(final.groupby(["Tipo_correspondencia", "Confianca"]).size().to_string(), "\n")

sem = final[final["Tipo_correspondencia"] == "SEM CORRESPONDÊNCIA"]["CARD_ID"].tolist()
if sem:
    print("IDs SEM correspondência:", *sem, sep="\n  - ")
revisar = final[final["Confianca"].isin(["Média", "Baixa"])]
if len(revisar):
    print("\nIDs para revisar manualmente (confiança Média/Baixa):")
    print(revisar[["CARD_ID", "ARO Name", "Confianca", "Observacao"]].to_string(index=False))

# --------------------------------- saída ------------------------------------
saida = final if INCLUIR_COLUNAS_AUDITORIA else final[["CARD_ID"] + COLUNAS_METADADOS]
saida.to_csv(ARQ_SAIDA, index=False, sep=SEP_SAIDA, encoding="utf-8-sig")
print(f"\nArquivo gerado: {ARQ_SAIDA} ({len(saida)} linhas, {saida['CARD_ID'].nunique()} IDs distintos)")

try:
    from google.colab import files
    files.download(ARQ_SAIDA)
except ImportError:
    pass


_IncompleteInputError: incomplete input (408986890.py, line 10)

In [ ]:
''' #REMOVA PARA VOLTAR O CÓDIGO ABAIXO
%cd /content/drive/MyDrive/Colab Notebooks/ResistoMap/CARD Dataset (Referência)/Metadados CARD final

df = pd.read_csv("metadados_card_final.csv") #Arquivos do CARD (Referência) só estão presentes em FASTA, foi necessário buscar os metadados

print(df.shape)
print(df.columns)
print(df.head())
print(df.dtypes)
print(df.isna().sum())


_IncompleteInputError: incomplete input (824809242.py, line 1)

In [ ]:
''' #REMOVA PARA VOLTAR O CÓDIGO ABAIXO

# Para evitar conflitos de variáveis, carregamos o dataset do NCBI em uma variável específica
card_df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/ResistoMap/NCBI Salmonella Enterica/NCBI dataset isolates.csv")

# 1. Espécie (Organism group)
print("=== Organism group ===")
print(card_df["#Organism group"].value_counts(dropna=False))

# 2. País (Location)
print("\n=== Location (Country) ===")
print(card_df["Location"].value_counts(dropna=False))

# 3. Ano (Extraído de 'Create date')
card_df['Year'] = card_df['Create date'].astype(str).str[:4]
print("\n=== Year ===")
print(card_df['Year'].value_counts(dropna=False))

# 4. Fonte de Isolamento (Isolation source ou Isolation type)
print("\n=== Isolation source ===")
print(card_df["Isolation source"].value_counts(dropna=False))

_IncompleteInputError: incomplete input (1947697823.py, line 1)

* Leitura Metadados CARD Final

In [ ]:
"""
Leitura e consolidação dos metadados do CARD para uso como
referência no projeto (genes de resistência em Salmonella enterica / S. bongori).

Fontes (pacote "card-data" de https://card.mcmaster.ca/download):
  - aro_index.tsv                                 -> TODAS as 12 colunas são mantidas
  - nucleotide_fasta_protein_homolog_model.fasta  -> cabeçalhos (organismo, fita,
                                                     coordenadas, nome) + comprimento

Saídas (pasta --saida):
  - referencia_card_completa.csv    tabela de referência completa (sem descartar colunas)
  - referencia_card_salmonella.csv  subconjunto marcado como Salmonella
  - referencia_card_manifest.json   rastreabilidade (hash dos insumos, contagens, data)

Uso no Colab (célula única):   exec(open("card_reference.py").read())   ou cole o código.
Uso como módulo:
    from card_reference import carregar_referencia, CardReference
    ref = CardReference(carregar_referencia("aro_index.tsv", "nucleotide_fasta_....fasta"))
    ref.por_aro("ARO:3002691");  ref.por_accession("AAL21173.1");  ref.salmonella()
"""

%cd /content/drive/MyDrive/Colab Notebooks/ResistoMap/CARD Dataset (Referência)/Cruzamento de IDs extraídos

# =============================== CONFIGURAÇÃO ===============================
ARQ_INDEX = "aro_index.tsv"
ARQ_FASTA = "nucleotide_fasta_protein_homolog_model.fasta"
PASTA_SAIDA = "."
VERSAO_CARD = ""            # preencha (ex.: "4.0.1") - o aro_index/FASTA não informam a versão
INCLUIR_SEQUENCIA = False   # True = coluna 'sequencia' no CSV (arquivo bem maior)
ENCODING_SAIDA = "utf-8-sig"
# ============================================================================

COLUNAS_INDEX_ESPERADAS = [
    "ARO Accession", "CVTERM ID", "Model Sequence ID", "Model ID", "Model Name",
    "ARO Name", "Protein Accession", "DNA Accession", "AMR Gene Family",
    "Drug Class", "Resistance Mechanism", "CARD Short Name",
]
RE_INTERVALO = re.compile(r"\d+-\d+")
RE_NOME_ORG = re.compile(r"^(.*)\[([^\]]+)\]\s*(.*)$")   # último [colchete] = organismo


# ------------------------------- utilidades ---------------------------------
def sha256(caminho, bloco=1 << 20):
    h = hashlib.sha256()
    with open(caminho, "rb") as fh:
        for parte in iter(lambda: fh.read(bloco), b""):
            h.update(parte)
    return h.hexdigest()


def normalizar_aro(valor):
    """'3002691' ou 'ARO:3002691' -> 'ARO:3002691'."""
    valor = str(valor).strip()
    return valor if valor.startswith("ARO:") else f"ARO:{valor}"


def sem_versao(acc):
    return str(acc).split(".")[0]


def classificar_salmonella(texto):
    """Retorna o rótulo de táxon encontrado em um texto, ou ''."""
    if not isinstance(texto, str) or "Salmonella" not in texto:
        return ""
    if re.search(r"Salmonella\s+enterica", texto):
        return "Salmonella enterica"
    if re.search(r"Salmonella\s+bongori", texto):
        return "Salmonella bongori"
    return "Salmonella (espécie não especificada)"


# --------------------------------- leitura ----------------------------------
def ler_aro_index(caminho):
    """Lê o aro_index.tsv inteiro, como texto (nada é convertido nem descartado)."""
    df = pd.read_csv(caminho, sep="\t", dtype=str, keep_default_na=False)
    df.columns = [c.strip() for c in df.columns]
    faltando = [c for c in COLUNAS_INDEX_ESPERADAS if c not in df.columns]
    if faltando:
        raise ValueError(f"{caminho}: colunas esperadas ausentes: {faltando}")
    return df


def ler_fasta(caminho, incluir_sequencia=False):
    """Lê o FASTA em streaming. Cabeçalho CARD:
         >gb|ACESSO|FITA|INICIO-FIM|ARO:xxxxxxx|Nome [Organismo] [Sufixo]
    Retorna (DataFrame, lista de cabeçalhos fora do padrão)."""
    registros, fora_padrao = [], []
    atual, comp, seq = None, 0, []

    def fechar():
        if atual is not None:
            atual["comprimento_pb"] = comp
            if incluir_sequencia:
                atual["sequencia"] = "".join(seq)
            registros.append(atual)

    with open(caminho, encoding="utf-8", errors="replace") as fh:
        for linha in fh:
            if linha.startswith(">"):
                fechar()
                comp, seq = 0, []
                cab = linha[1:].rstrip("\r\n")
                p = cab.split("|", 5)
                if len(p) == 6 and RE_INTERVALO.fullmatch(p[3]) and p[4].startswith("ARO:"):
                    ini, fim = p[3].split("-")
                    m = RE_NOME_ORG.match(p[5])
                    nome, org, suf = (m.group(1).strip(), m.group(2).strip(), m.group(3).strip()) \
                        if m else (p[5].strip(), "", "")
                    atual = {
                        "fasta_accession": p[1], "fasta_fita": p[2],
                        "fasta_inicio": int(ini), "fasta_fim": int(fim),
                        "fasta_aro": p[4], "fasta_nome": nome,
                        "fasta_organismo": org, "fasta_sufixo": suf,
                        "fasta_header": cab,
                    }
                else:
                    fora_padrao.append(cab)
                    atual = {"fasta_header": cab}
            elif atual is not None:
                s = linha.strip()
                comp += len(s)
                if incluir_sequencia:
                    seq.append(s)
    fechar()
    df = pd.DataFrame(registros)
    for c in ("fasta_inicio", "fasta_fim"):
        df[c] = pd.to_numeric(df[c], errors="coerce").astype("Int64")
    return df, fora_padrao


# ------------------------------- consolidação -------------------------------
def carregar_referencia(caminho_index=ARQ_INDEX, caminho_fasta=ARQ_FASTA,
                        incluir_sequencia=INCLUIR_SEQUENCIA, verbose=True):
    """Une aro_index + FASTA em uma tabela única, sem perder nenhuma linha de
    nenhum dos dois arquivos (coluna 'origem' indica de onde veio cada linha)."""
    idx = ler_aro_index(caminho_index)
    fasta, fora_padrao = ler_fasta(caminho_fasta, incluir_sequencia)
    idx["_idx_row"] = range(len(idx))
    fasta["_fasta_row"] = range(len(fasta))

    # chave = ARO Accession. Se um ARO tem >1 modelo no índice, prefere-se o
    # modelo cujo DNA Accession coincide com o accession do FASTA.
    m = fasta.merge(idx, left_on="fasta_aro", right_on="ARO Accession", how="left")
    m["acc_consistente"] = m["fasta_accession"] == m["DNA Accession"]
    m["n_modelos_index_por_aro"] = m.groupby("_fasta_row")["_idx_row"].transform("count")
    tem_ok = m.groupby("_fasta_row")["acc_consistente"].transform("any")
    m = m[~(tem_ok & ~m["acc_consistente"])].copy()
    m["origem"] = m["_idx_row"].notna().map({True: "aro_index+fasta", False: "so_fasta"})

    so_idx = idx[~idx["_idx_row"].isin(m["_idx_row"].dropna())].copy()
    so_idx["origem"] = "so_aro_index"
    so_idx["n_modelos_index_por_aro"] = so_idx.groupby("ARO Accession")["_idx_row"].transform("count")

    ref = pd.concat([m, so_idx], ignore_index=True)
    ref["acc_consistente"] = ref["acc_consistente"].where(ref["origem"] == "aro_index+fasta")
    ref["comprimento_pb"] = ref["comprimento_pb"].astype("Int64")
    ref["fasta_ambiguo"] = (ref["origem"] == "aro_index+fasta") & (ref["n_modelos_index_por_aro"] > 1)

    # Salmonella: organismo do FASTA e/ou nome do modelo (ARO Name) no índice
    t_org = ref["fasta_organismo"].map(classificar_salmonella)
    t_nome = ref["ARO Name"].map(classificar_salmonella)
    ref["salmonella_taxon"] = t_org.where(t_org != "", t_nome)
    ref["salmonella_fonte"] = ""
    ref.loc[t_nome != "", "salmonella_fonte"] = "ARO Name"
    ref.loc[t_org != "", "salmonella_fonte"] = "organismo_fasta"
    ref.loc[(t_org != "") & (t_nome != ""), "salmonella_fonte"] = "organismo_fasta+ARO Name"
    ref["eh_salmonella"] = ref["salmonella_taxon"] != ""

    # coordenadas em 1-based (CARD guarda início 0-based; validado em relatorio())
    ref["fasta_inicio_1based"] = ref["fasta_inicio"] + 1

    ordem = (COLUNAS_INDEX_ESPERADAS
             + ["fasta_accession", "fasta_fita", "fasta_inicio", "fasta_inicio_1based", "fasta_fim",
                "comprimento_pb", "fasta_aro", "fasta_nome", "fasta_organismo", "fasta_sufixo",
                "fasta_header", "origem", "acc_consistente", "n_modelos_index_por_aro",
                "fasta_ambiguo", "eh_salmonella", "salmonella_taxon", "salmonella_fonte"]
             + (["sequencia"] if incluir_sequencia else []))
    extras = [c for c in idx.columns if c not in ordem and not c.startswith("_")]
    ref = ref[ordem + extras]
    # ARO de linhas 'so_fasta' (sem índice) permanece em fasta_aro; preenche a chave
    ref["ARO Accession"] = ref["ARO Accession"].where(ref["ARO Accession"].notna(), ref["fasta_aro"])
    ref = ref.fillna({c: "" for c in COLUNAS_INDEX_ESPERADAS + ["fasta_accession", "fasta_fita",
        "fasta_aro", "fasta_nome", "fasta_organismo", "fasta_sufixo", "fasta_header"]})
    ref.attrs["fora_padrao"] = fora_padrao
    ref.attrs["n_idx"], ref.attrs["n_fasta"] = len(idx), len(fasta)
    if verbose:
        relatorio(ref)
    return ref.reset_index(drop=True)


def relatorio(ref):
    """Validações e contagens para conferir a carga."""
    print("=" * 70)
    print(f"linhas do aro_index: {ref.attrs.get('n_idx')} | registros no FASTA: {ref.attrs.get('n_fasta')}")
    print(f"linhas na referência: {len(ref)}")
    print(ref["origem"].value_counts().to_string(), "\n")
    if ref.attrs.get("fora_padrao"):
        print(f"AVISO: {len(ref.attrs['fora_padrao'])} cabeçalho(s) fora do padrão esperado.")
    aros_fasta = set(ref.loc[ref["origem"] != "so_aro_index", "ARO Accession"])
    preteridos = ref[(ref["origem"] == "so_aro_index") & ref["ARO Accession"].isin(aros_fasta)]
    print(f"modelos do índice com ARO presente no FASTA, mas accession diferente do FASTA "
          f"(mantidos como so_aro_index): {len(preteridos)}")
    print(f"linhas ambíguas (registro do FASTA ligado a >1 modelo do índice, mesmo accession): "
          f"{int(ref['fasta_ambiguo'].sum())}")
    print(f"registros do FASTA sem organismo entre colchetes: "
          f"{int(((ref['origem'] != 'so_aro_index') & (ref['fasta_organismo'] == '')).sum())}")
    com_seq = ref[ref["comprimento_pb"].notna() & ref["fasta_inicio"].notna()]
    if len(com_seq):
        ok = ((com_seq["fasta_fim"] - com_seq["fasta_inicio"]) == com_seq["comprimento_pb"]).mean()
        print(f"coordenada (fim - início) == comprimento da sequência: {ok:.1%} "
              "(confirma início 0-based, fim inclusivo)")
    print("\nSalmonella na referência:")
    sal = ref[ref["eh_salmonella"]]
    print(sal["salmonella_taxon"].value_counts().to_string() if len(sal) else "  nenhuma")
    if not (ref["salmonella_taxon"] == "Salmonella bongori").any():
        print("AVISO: nenhum registro de Salmonella bongori nestes dois arquivos "
              "(organismo do FASTA ou ARO Name).")
    print("=" * 70)


# ------------------------------- consultas ----------------------------------
class CardReference:
    """Consultas sobre a tabela de referência."""

    def __init__(self, df):
        self.df = df

    def por_aro(self, aro):
        return self.df[self.df["ARO Accession"] == normalizar_aro(aro)]

    def por_accession(self, acc):
        """Procura em Protein Accession, DNA Accession e accession do FASTA;
        se não achar, tenta sem o sufixo de versão."""
        d = self.df
        m = d[(d["Protein Accession"] == acc) | (d["DNA Accession"] == acc) | (d["fasta_accession"] == acc)]
        if m.empty:
            b = sem_versao(acc)
            m = d[(d["Protein Accession"].map(sem_versao) == b) | (d["DNA Accession"].map(sem_versao) == b)
                  | (d["fasta_accession"].map(sem_versao) == b)]
        return m

    def por_coordenada(self, acc, inicio, fim, tolerancia=1):
        """Casa intervalo (0- ou 1-based no início) com o FASTA."""
        d = self.df
        return d[(d["fasta_accession"] == acc)
                 & ((d["fasta_inicio"] - inicio).abs() <= tolerancia) & (d["fasta_fim"] == fim)]

    def por_id_card(self, cid):
        """Aceita IDs como 'gb|AAL21173.1|Salmonella' ou 'gb|AJ487033.2|1686-2858|Salmonella'."""
        p = [x.strip() for x in cid.split("|")]
        acc = p[1] if len(p) > 1 else p[0]
        rng = next((x for x in p if RE_INTERVALO.fullmatch(x)), "")
        if rng:
            ini, fim = map(int, rng.split("-"))
            m = self.por_coordenada(acc, ini, fim)
            if len(m):
                return m
        return self.por_accession(acc)

    def salmonella(self, taxon=None):
        d = self.df[self.df["eh_salmonella"]]
        return d if taxon is None else d[d["salmonella_taxon"].str.contains(taxon, case=False)]


# --------------------------------- saída ------------------------------------
def salvar(ref, pasta=PASTA_SAIDA, versao_card=VERSAO_CARD, caminho_index=ARQ_INDEX,
           caminho_fasta=ARQ_FASTA):
    os.makedirs(pasta, exist_ok=True)
    f_comp = os.path.join(pasta, "referencia_card_completa.csv")
    f_sal = os.path.join(pasta, "referencia_card_salmonella.csv")
    f_man = os.path.join(pasta, "referencia_card_manifest.json")
    ref.to_csv(f_comp, index=False, encoding=ENCODING_SAIDA)
    ref[ref["eh_salmonella"]].to_csv(f_sal, index=False, encoding=ENCODING_SAIDA)
    manifest = {
        "gerado_em": datetime.now().isoformat(timespec="seconds"),
        "versao_card_informada": versao_card or "NÃO INFORMADA",
        "insumos": {os.path.basename(p): {"sha256": sha256(p), "bytes": os.path.getsize(p)}
                    for p in (caminho_index, caminho_fasta)},
        "linhas_aro_index": ref.attrs.get("n_idx"),
        "registros_fasta": ref.attrs.get("n_fasta"),
        "linhas_referencia": len(ref),
        "por_origem": ref["origem"].value_counts().to_dict(),
        "salmonella_por_taxon": ref[ref["eh_salmonella"]]["salmonella_taxon"].value_counts().to_dict(),
    }
    with open(f_man, "w", encoding="utf-8") as fh:
        json.dump(manifest, fh, ensure_ascii=False, indent=2)
    return f_comp, f_sal, f_man


def garantir_arquivos(caminhos):
    """No Colab, abre o upload caso algum arquivo falte."""
    faltando = [c for c in caminhos if not os.path.exists(c)]
    if not faltando:
        return
    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError(f"Arquivos não encontrados: {faltando}")
    print("Envie os arquivos:", faltando)
    files.upload()
    faltando = [c for c in faltando if not os.path.exists(c)]
    if faltando:
        raise FileNotFoundError(f"Ainda faltam: {faltando}")


def main(argv=None):
    ap = argparse.ArgumentParser(description="Consolida a referência CARD (aro_index + FASTA).")
    ap.add_argument("--index", default=ARQ_INDEX)
    ap.add_argument("--fasta", default=ARQ_FASTA)
    ap.add_argument("--saida", default=PASTA_SAIDA)
    ap.add_argument("--versao-card", default=VERSAO_CARD)
    ap.add_argument("--com-sequencia", action="store_true", default=INCLUIR_SEQUENCIA)
    args, _ = ap.parse_known_args(argv)   # parse_known_args: tolera os argumentos do Colab/Jupyter
    garantir_arquivos([args.index, args.fasta])
    ref = carregar_referencia(args.index, args.fasta, args.com_sequencia)
    arquivos = salvar(ref, args.saida, args.versao_card, args.index, args.fasta)
    print("Arquivos gerados:", *arquivos, sep="\n  - ")
    if not args.versao_card:
        print("\nAviso: informe --versao-card (ou VERSAO_CARD) para registrar a versão do CARD no manifest.")
    return ref


if __name__ == "__main__":
    referencia = main(sys.argv[1:])

/content/drive/MyDrive/Colab Notebooks/ResistoMap/CARD Dataset (Referência)/Cruzamento de IDs extraídos
linhas do aro_index: 6463 | registros no FASTA: 6059
linhas na referência: 6463
origem
aro_index+fasta    6059
so_aro_index        404 

modelos do índice com ARO presente no FASTA, mas accession diferente do FASTA (mantidos como so_aro_index): 2
linhas ambíguas (registro do FASTA ligado a >1 modelo do índice, mesmo accession): 2
registros do FASTA sem organismo entre colchetes: 0
coordenada (fim - início) == comprimento da sequência: 100.0% (confirma início 0-based, fim inclusivo)

Salmonella na referência:
salmonella_taxon
Salmonella enterica                      79
Salmonella (espécie não especificada)     8
AVISO: nenhum registro de Salmonella bongori nestes dois arquivos (organismo do FASTA ou ARO Name).
Arquivos gerados:
  - ./referencia_card_completa.csv
  - ./referencia_card_salmonella.csv
  - ./referencia_card_manifest.json

Aviso: informe --versao-card (ou VERSAO_CARD) par

______________________________________________________

# Leitura Metadados/CSV NCBI

In [ ]:
%cd /content/drive/MyDrive/Colab Notebooks/ResistoMap/NCBI Salmonella Enterica

df = pd.read_csv("NCBI dataset isolates.csv")

print(df.shape)
print(df.columns)
print(df.head())
print(df.dtypes)
print(df.isna().sum())


/content/drive/MyDrive/Colab Notebooks/ResistoMap/NCBI Salmonella Enterica
(4783, 17)
Index(['#Organism group', 'Strain', 'Isolate identifiers', 'Serovar',
       'Isolate', 'Create date', 'Location', 'Isolation source',
       'Isolation type', 'Food origin', 'SNP cluster', 'Min-same', 'Min-diff',
       'BioSample', 'Assembly', 'AMR genotypes', 'Computed types'],
      dtype='object')
       #Organism group       Strain  \
0  Salmonella enterica   IVB 126/82   
1  Salmonella enterica   IVB 176/82   
2  Salmonella enterica       287/91   
3  Salmonella enterica  FDAARGOS_54   
4  Salmonella enterica      BR_RS12   

                                 Isolate identifiers      Serovar  \
0  "BCW_1789","CFSAN000575","IVB 126/82","SRS535828"   Heidelberg   
1  "BCW_1843","CFSAN000164","IVB 176/82","SRS535853"  Enteritidis   
2        "001455_GCAT-20090507","287/91","ERS000073"   Gallinarum   
3        "FDA:FDAARGOS_54","FDAARGOS_54","SRS891668"   Heidelberg   
4                             

In [ ]:

# 1. Espécie (Organism group)
print("=== Organism group ===")
print(df["#Organism group"].value_counts(dropna=False))

# 2. País (Location)
print("\n=== Location (Country) ===")
print(df["Location"].value_counts(dropna=False))

# 3. Ano (Extraído de 'Create date')
# Como o Create date está no formato YYYY-MM-DD..., podemos extrair os 4 primeiros caracteres para obter o ano
df['Year'] = df['Create date'].astype(str).str[:4]
print("\n=== Year ===")
print(df['Year'].value_counts(dropna=False))

# 4. Fonte de Isolamento (Isolation source ou Isolation type)
print("\n=== Isolation source ===")
print(df["Isolation source"].value_counts(dropna=False))


=== Organism group ===
#Organism group
Salmonella enterica    4783
Name: count, dtype: int64

=== Location (Country) ===
Location
Brazil                            4218
Brazil: Sao Paulo, Sao Paulo       161
Brazil: Sao Paulo, Avare            53
Brazil: Parana                      46
Brazil: Parana state                38
                                  ... 
Brazil: Mato Grosso, Aripuana        1
Brazil: Sao Paulo, Americana         1
Brazil: Sao Paulo, Botucatu          1
Brazil: SC                           1
Brazil: Sao Paulo, Jaboticabal       1
Name: count, Length: 74, dtype: int64

=== Year ===
Year
2023    967
2024    951
2022    945
2026    780
2025    441
2018    199
2021    167
2019    108
2016     95
2017     76
2020     52
2015      2
Name: count, dtype: int64

=== Isolation source ===
Isolation source
fresh water stream    991
dam water             896
NaN                   332
Stream                315
water                 205
                     ... 
Anal swab      

______________________________________________________

# Filtrar genomas com base nos critérios:

In [ ]:
'''
#Completude do genoma
completeness >= 70/100 #70%
completeness >= 80/100 #80% de completude no genoma

#Nível de contaminação
contamination <= 5/100

#Contagem de contigs
n_contigs <= 200

df["completeness"].describe()
df["contamination"].describe()
df["n_contigs"].describe()
'''

In [ ]:
print("Colunas disponíveis:", list(df.columns)) #Verifica quais colunas/critérios estão disponíveis no arquivo NCBI dataset isolates.csv

In [ ]:
# Com base na saída anterior
# O arquivo 'NCBI dataset isolates.csv' não contém as métricas de completude, contaminação ou contigs.
# Para utilizá-las, vamos precisar carregar e cruzar esses dados a partir de outra tabela de controle de qualidade.

# Exemplo ilustrativo de como o código deveria ser se as colunas existissem...:
"""
# Filtrar genomas com base nos critérios:
completo_80 = df[df["completeness"] >= 80]
contaminacao_baixa = df[df["contamination"] <= 5]
contigs_filtrados = df[df["n_contigs"] <= 200]

# Exibir estatísticas descritivas
print(df["completeness"].describe())
print(df["contamination"].describe())
print(df["n_contigs"].describe())
"""

print("Colunas disponíveis:", list(df.columns)) #Deixei novamente o código aqui para relembrar

In [ ]:
'''
%cd /content/drive/MyDrive/Colab Notebooks/ResistoMap

!head -n 20 GCA_001188125.2.FASTA
'''

In [ ]:
'''
fasta_path = "data/genomes/SALM_001.fasta"
'''

In [ ]:
'''
records = list(SeqIO.parse(fasta_path, "fasta"))
'''